In [22]:
import pandas as pd
from pathlib import Path
import numpy as np

DF_PATH = Path("../data/raw/openaq.csv")
df = pd.read_csv(DF_PATH, delimiter=";")

In [23]:
# look at the df
df.head()

,Country Code,City,Location,Coordinates,Pollutant,Source Name,Unit,Value,Last Updated,Country Label
0,BE,NaN,Escautpont,"50.420270857658636, 3.551812869268811",SO2,EEA France,µg/m³,3.60000,2017-07-18T22:00:00+02:00,Belgium
1,BG,Teleorman-RNMCA,NET-RO058A,"43.650721999999995, 25.363583",CO,EEA Romania,µg/m³,1237.25114,2024-03-11T09:00:00+01:00,Bulgaria
2,BG,National air network,NET-BG001A,"42.518891999999994, 27.375144",O3,EEA Bulgaria,µg/m³,12.06000,2024-03-11T08:00:00+01:00,Bulgaria
3,BG,National air network,NET-BG001A,"42.669796999999996, 23.268403000000003",NO,EEA Bulgaria,µg/m³,42.43000,2024-03-11T08:00:00+01:00,Bulgaria
4,BG,National air network,NET-BG001A,"43.217279999999995, 27.935959999999998",NO2,EEA Bulgaria,µg/m³,16.60000,2024-03-11T08:00:00+01:00,Bulgaria


## Cleaning

In [24]:
# shape
df.shape

(54551, 10)

In [25]:
# size
df.size

545510

In [26]:
# info about the df
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 54551 entries, 0 to 54550
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Country Code   54551 non-null  str    
 1   City           31021 non-null  str    
 2   Location       54549 non-null  str    
 3   Coordinates    54361 non-null  str    
 4   Pollutant      54551 non-null  str    
 5   Source Name    54551 non-null  str    
 6   Unit           54551 non-null  str    
 7   Value          54551 non-null  float64
 8   Last Updated   54551 non-null  str    
 9   Country Label  54436 non-null  str    
dtypes: float64(1), str(9)
memory usage: 9.6 MB


In [27]:
# check NaN values
df.isna().sum()

Country Code         0
City             23530
Location             2
Coordinates        190
Pollutant            0
Source Name          0
Unit                 0
Value                0
Last Updated         0
Country Label      115
dtype: int64

In [28]:
# fill categorical features' missing values with Unknown
cat_features_missing_vals = ["City", "Location", "Country Label"]

for cat_feature_missing_vals in cat_features_missing_vals:
    df[cat_feature_missing_vals] = df[cat_feature_missing_vals].replace(np.nan, "Unknown")

In [29]:
# let's see NaN now
df.isna().sum()

# we can leave some Coordinates data as NaN, instead of replacing it, preserving that data as it is.
# some models handle missing values well

Country Code       0
City               0
Location           0
Coordinates      190
Pollutant          0
Source Name        0
Unit               0
Value              0
Last Updated       0
Country Label      0
dtype: int64

In [30]:
# shape
df.shape

(54551, 10)

In [31]:
# duplicate values?
df.duplicated().sum()

np.int64(0)

In [32]:
# change some str columns to category for better performance
df[["Country Code", "Pollutant", "Unit", "Country Label"]] = df[["Country Code", "Pollutant", "Unit", "Country Label"]].astype("category")

In [33]:
# change Last Updated to datetime format
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce", utc=True)

In [34]:
# move target feature to the end
col_to_move = df.pop("Value")
df.insert(len(df.columns), "Value", col_to_move)

In [35]:
# clean negative Value values because pollution can't be negative
df = df[df.Value >= 0]

In [36]:
# drop rows if Value reaches too high numbers (for ex. more than 1 million)
# keep it only if the pollutant is UM003
invalid_values = ((df.Value > 1000000) & (df.Pollutant != "UM003"))
df = df[~invalid_values]

In [38]:
# check data types
df.dtypes

Country Code                category
City                             str
Location                         str
Coordinates                      str
Pollutant                   category
Source Name                      str
Unit                        category
Last Updated     datetime64[us, UTC]
Country Label               category
Value                        float64
dtype: object

In [39]:
# see final dataframe
df.head()

,Country Code,City,Location,Coordinates,Pollutant,Source Name,Unit,Last Updated,Country Label,Value
0,BE,Unknown,Escautpont,"50.420270857658636, 3.551812869268811",SO2,EEA France,µg/m³,2017-07-18 20:00:00+00:00,Belgium,3.60000
1,BG,Teleorman-RNMCA,NET-RO058A,"43.650721999999995, 25.363583",CO,EEA Romania,µg/m³,2024-03-11 08:00:00+00:00,Bulgaria,1237.25114
2,BG,National air network,NET-BG001A,"42.518891999999994, 27.375144",O3,EEA Bulgaria,µg/m³,2024-03-11 07:00:00+00:00,Bulgaria,12.06000
3,BG,National air network,NET-BG001A,"42.669796999999996, 23.268403000000003",NO,EEA Bulgaria,µg/m³,2024-03-11 07:00:00+00:00,Bulgaria,42.43000
4,BG,National air network,NET-BG001A,"43.217279999999995, 27.935959999999998",NO2,EEA Bulgaria,µg/m³,2024-03-11 07:00:00+00:00,Bulgaria,16.60000


In [41]:
# save the dataframe
df.to_parquet("../data/processed/openaq_cleaned.parquet", index=False)